In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import sionna.rt as rt
from utils.scene_utils import add_txs
from utils.geo_coords import SceneCoordinateConverter
from config import FILE_ENCODE, LocalCRS

# Define the geographical boundary of the target region:
# I take the max and min latitude and longitude of transmitters and extend it with 500m
# Extended Formula:
# latitude 1° is approximately equal to 111,000m
# 500 / 111 000 ≈ 0.004505
# 48.90138888888889 + 0.004505 ≈ 48.9059; 48.818333333333335 - 0.004505 ≈ 48.8138
# longitude 1° is approximately equal to $\Delta_{lon} =\Delta_{lat} \times \cos(lat)$
# 111 000 * cos((48.9014 + 48.8183)/2) ≈ 111 000 * cos(48.86) ≈ 111 000 * 0.6579 ≈ 73 000
# 500 / 73 000 ≈ 0.006849
# 2.249722222222222 - 0.006849 ≈ 2.2429 ; 2.450555555555556 + 0.006849 ≈ 2.4574
LAT_MAX, LAT_MIN = 48.9059, 48.8138
LON_MIN, LON_MAX = 2.2429, 2.4574

# Calculate the center origin point of the scene
LAT_ORIGIN=(LAT_MAX + LAT_MIN)/2
LON_ORIGIN=(LON_MIN + LON_MAX)/2
TRANSMITTER_DIRECTORY = Path('data/transmitters')

DATASET_DIR = Path('data/dataset')  # Root directory for HDF5 block files

In [ ]:
# Optional: Preprocess antenna files. Filter antennas by frequence and postal code
from utils.preprocess_raw_data import preprocess_antenna_data_by_frequency_and_postal

ANTENNES_INFO_FILENAME = Path('data/paris/Antennes_Emetteurs_Bandes_Cartoradio.csv')
ANTENNES_LOC_FILENAME = Path('data/paris/Sites_Cartoradio.csv')
FILTER_FREQUENCE = 2600
FILTER_POSTAL_CODE = r'^75\d{3}$'

preprocess_antenna_data_by_frequency_and_postal(
    ANTENNES_INFO_FILENAME, 
    ANTENNES_LOC_FILENAME, 
    FILTER_FREQUENCE, 
    FILTER_POSTAL_CODE, 
    TRANSMITTER_DIRECTORY)

# Step 1: Region Grid Splitting
In this step, we split the large Paris area into smaller blocks (e.g., 2.56km x 2.56km). 
This helps our computer run Sionna simulation without running out of memory.

**Key features:**
* Use official French national projection - **Lambert-93** to work with meters.
* Add an **overlap buffer** (e.g., 150m) to avoid edge errors in ray-tracing.

In [ ]:
# Import the splitter class from map_splitter.py
from utils.map_splitter import TileSplitter

# Initialize the splitter
# Each block is 2560m x 2560m. Overlap is 150m.
splitter = TileSplitter(
    lat_min=LAT_MIN, 
    lat_max=LAT_MAX, 
    lon_min=LON_MIN, 
    lon_max=LON_MAX, 
    block_size_m=2560, 
    overlap_m=150
)

# Get the task list for loops later
all_blocks = splitter.get_all_blocks()

# Step 2: Per-Block PLY & XML Generation

In this step, we iterate over each grid block and generate the 3D scene files required by Sionna ray-tracing. For each block we produce **all available layer** PLY mesh files (buildings, roads, railways, water, forest) plus terrain, and one Mitsuba XML scene file.

**Key features:**
* Fetch OSM data per block using `OSMFetcher` with the block's **overlap-extended** bounding box, ensuring consistent coverage with the terrain.
* Convert OSM layer footprints (buildings, roads, railways, water, forest) to 3D meshes via `OSMToPLY`. Buildings **without height information are dropped** to avoid unrealistic flat geometry; other layers use a default height of `0.0`.
* Generate a flat terrain mesh at `z = 0.0` with **10 m resolution** covering the full overlap-extended Lambert-93 extent.
* Place all PLY files under a `meshes/` subdirectory alongside the XML, matching Sionna's expected path convention (`meshes/<filename>.ply`).
* Assemble the final XML scene with `SionnaXMLGenerator`, referencing all available mesh layers. Layers with no features still produce a minimal valid PLY file, keeping the XML structure consistent across blocks.

In [ ]:
from pathlib import Path

XML_DIR = Path('data/xml')  # Root directory for all block outputs
TERRAIN_RESOLUTION = 10.0   # Grid cell size in meters
TERRAIN_HEIGHT = 0.0        # Constant Z height for terrain vertices

# =============================================================================
# Layer Definitions
# Each layer maps an OSM tag set to its output PLY filename and default height.
# Buildings use 'drop' mode: polygons without height info are skipped.
# Other layers use default_height=0.0 (flat geometry at ground level).
# =============================================================================
LAYERS = [
    {
        "tag_name": "buildings",
        "ply_filename": "buildings.ply",
        "default_height": 0.0,       # Not used for buildings (see handle_missing_height)
        "handle_missing_height": "drop",
    },
    {
        "tag_name": "roads",
        "ply_filename": "roads.ply",
        "default_height": 0.1,
        "handle_missing_height": "use_default",
    },
    {
        "tag_name": "railways",
        "ply_filename": "railways.ply",
        "default_height": 0.5,
        "handle_missing_height": "use_default",
    },
    {
        "tag_name": "water",
        "ply_filename": "water.ply",
        "default_height": 0.2,
        "handle_missing_height": "use_default",
    },
    {
        "tag_name": "forest",
        "ply_filename": "forest.ply",
        "default_height": 2.0,
        "handle_missing_height": "use_default",
    },
]

In [ ]:
from utils.osm_fetcher import OSMFetcher, OSM_TAGS
from utils.osm_to_ply import OSMToPLY
from utils.osm_to_ply import generate_flat_terrain_ply
from utils.generate_xml import SionnaXMLGenerator

for block_info in all_blocks:
    row = block_info["row"]
    col = block_info["col"]
    block_name = block_info["name"]
    
    print(f"\n{'='*60}")
    print(f"Processing {block_name} (row={row}, col={col})")
    print(f"{'='*60}")
    
    # --- Get block metadata (with overlap for consistent coverage) ---
    meta = splitter.get_block_latlon_bounds(row, col)
    
    # --- Create output directories ---
    block_dir = XML_DIR / block_name
    mesh_dir = block_dir / "meshes"
    mesh_dir.mkdir(parents=True, exist_ok=True)
    
    # --- Fetch OSM data for this block ---
    # IMPORTANT: bbox order is (left, bottom, right, top)
    # which corresponds to (lon_min, lat_min, lon_max, lat_max)
    fetcher = OSMFetcher(
        bbox=(meta.lon_min, meta.lat_min, meta.lon_max, meta.lat_max),
        tags=OSM_TAGS["complete"],
    )
    
    # --- Generate PLY for each layer ---
    for layer in LAYERS:
        tag_name = layer["tag_name"]
        ply_filename = layer["ply_filename"]
        ply_path = mesh_dir / ply_filename
        
        # Filter the pre-fetched data for this specific layer
        gdf = fetcher.get_filtered_features(OSM_TAGS[tag_name])
        
        if gdf.empty:
            print(f"  [{tag_name}] No features found — writing empty PLY")
        
        # Convert OSM geometries to 3D PLY mesh
        converter = OSMToPLY(
            gdf=gdf,
            ply_path=ply_path,
            default_height=layer["default_height"],
            block_meta=meta,
        )
        converter._process_polygons(handle_missing_height=layer["handle_missing_height"])
        converter._collect_3d_polygons()
        converter._build_multi_polygon()
        converter.save_to_ply()
        
        print(f"  [{tag_name}] Saved to {ply_path}")
    
    # --- Generate terrain PLY ---
    # Uses the overlap-extended Lambert-93 coordinates to match OSM coverage
    terrain_path = mesh_dir / "terrain.ply"
    generate_flat_terrain_ply(
        output_path=terrain_path,
        x_min=meta.x_start,
        x_max=meta.x_end,
        y_min=meta.y_start,
        y_max=meta.y_end,
        resolution=TERRAIN_RESOLUTION,
        height=TERRAIN_HEIGHT,
        expand_m= meta.overlap_m*2
    )
    print(f"  [terrain] Saved to {terrain_path}")
    
    # --- Generate Sionna XML scene file ---
    xml_path = block_dir / f"{block_name}.xml"
    xml_generator = SionnaXMLGenerator(mesh_dir=mesh_dir, output_path=xml_path)
    xml_generator.generate(validate_meshes=False)  # Allow missing layers (empty PLYs exist)
    print(f"  [xml] Saved to {xml_path}")
    
    print(f"Finished {block_name}")

print(f"\n{'='*60}")
print(f"All {len(all_blocks)} blocks processed successfully.")
print(f"Output directory: {XML_DIR.resolve()}")

# Step 3: Per-Block Radio Map Generation

In this step, we load each block's Mitsuba XML scene into Sionna 2.0.1 and compute a summed RSS radio map using ray-tracing. For each block, transmitters inside the **core area** (overlap excluded) are filtered from a global CSV and placed into the scene. The resulting radio map captures the combined received signal strength from all active transmitters over a dense 1 m grid.

**Key features:**
* Initialize a `SceneCoordinateConverter` per block with its origin at the **center of the overlap-extended bounding box** and altitude at `TERRAIN_HEIGHT`, matching the terrain surface.
* Load the block's XML scene via `rt.load_scene()` and configure it with a `FREQUENCY` carrier frequency and a single-element cross-polarized antenna array (`tr38901` pattern).
* Filter transmitters from `TRANSMITTER_PATH` to keep only those within the block's **core area** (no overlap), avoiding duplicate coverage across neighboring blocks.
* Update radio materials for water and roads to physically accurate ITU models (`freshwater`, `asphalt_concrete`) at the target frequency.
* Compute the radio map with `rt.RadioMapSolver` at `RM_RESOLUTION_M` over the full `block_size_m × block_size_m` core area, using up to 12 ray bounces (`max_depth=5`) and 10 million rays per transmitter.
* Sum the linear RSS contributions from all transmitters into a single 2D array of shape `(block_size_m, block_size_m)`. Blocks with **no transmitters** in their core area return `None` and are skipped with a log message.

In [ ]:
TRANSMITTER_PATH = Path('data/transmitters/2600_mhz.csv')  # Global transmitter file
FREQUENCY = 2.6e9          # Carrier frequency in Hz (2.6 GHz)
RM_RESOLUTION_M = 1.0  # Radio map cell size in meters

# =============================================================================
# Antenna Array
# Single-element cross-polarized antenna following TR 38.901 pattern.
# =============================================================================
tx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross",
)

In [ ]:
from config import LocalCRS
from utils.geo_coords import SceneCoordinateConverter
from utils.generate_radiomap import RadioMapGenerator
from utils.h5_manager import H5Manager

for block_info in all_blocks:
    row = block_info["row"]
    col = block_info["col"]
    block_name = block_info["name"]
    
    print(f"\n{'='*60}")
    print(f"Processing {block_name} (row={row}, col={col})")
    print(f"{'='*60}")
    
    # --- Get block metadata ---
    meta = splitter.get_block_latlon_bounds(row, col)
    
    # --- Paths for this block ---
    block_dir = XML_DIR / block_name
    xml_path = block_dir / f"{block_name}.xml"

    # --- H5 Init ---
    h5_path = DATASET_DIR / f"{block_name}.h5"
    h5_path.parent.mkdir(parents=True, exist_ok=True)
    H5Manager.init_block_file(h5_path, meta, RM_RESOLUTION_M)
    
    if not xml_path.exists():
        print(f"  [skip] XML file not found: {xml_path}")
        continue
    
    # --- Initialize coordinate converter for this block ---
    # Origin is the center of the overlap-extended bounding box, at z = TERRAIN_HEIGHT
    # (matching the terrain surface height used in Step 2).
    lat_origin = (meta.lat_min + meta.lat_max) / 2.0
    lon_origin = (meta.lon_min + meta.lon_max) / 2.0
    converter = SceneCoordinateConverter(
        lat_origin,
        lon_origin,
        TERRAIN_HEIGHT,                    # ALT_ORIGIN matches terrain height
        LocalCRS.OSM_STORAGE.crs,
        LocalCRS.FRANCE_LAMBERT93.crs,
    )
    
    # --- Generate radio map ---
    generator = RadioMapGenerator(converter)
    
    rss_map = generator.generate(
        xml_path=xml_path,
        csv_path=TRANSMITTER_PATH,
        block_meta=meta,
        tx_array=tx_array,
        frequency=FREQUENCY,
        resolution_m=RM_RESOLUTION_M,
    )
    
    # --- save result ---
    if rss_map is None:
        print(f"  [skip] No transmitters in core area of {block_name}")
        # Remove incomplete H5 file since radio map is essential
        if h5_path.exists():
            h5_path.unlink()
            print(f"  [h5] Removed {h5_path} (no radio map)")
    else:
        print(f"  [done] RSS map shape: {rss_map.shape}, "
              f"dtype: {rss_map.dtype}, "
              f"min: {rss_map.min():.6e}, max: {rss_map.max():.6e}")
        # Write radio map to HDF5
        H5Manager.write_dataset(
            h5_path,
            H5Manager.DATASET_RADIOMAP,
            rss_map,
            dtype='float32',
        )
        print(f"  [h5] Written radiomap to {h5_path}")

print(f"\n{'='*60}")
print(f"All {len(all_blocks)} blocks processed.")

# Step 4: Building Raster Generation

In this step, we rasterize building footprints into a binary presence matrix at the same resolution with radio map.

**Key features:**
* Re-fetch OSM building footprints for each block and rasterize them into a **binary presence matrix** (1 = building, 0 = open ground) at the same `RM_RESOLUTION_M`, with the overlap region cropped to match the core area extent.

In [ ]:
from utils.osm_fetcher import OSMFetcher, OSM_TAGS
from utils.building_rasterizer import BuildingRasterizer

for block_info in all_blocks:
    row = block_info["row"]
    col = block_info["col"]
    block_name = block_info["name"]
    
    print(f"\n{'='*60}")
    print(f"Processing {block_name} (row={row}, col={col})")
    print(f"{'='*60}")
    
    # --- Get block metadata ---
    meta = splitter.get_block_latlon_bounds(row, col)
    
    # --- Fetch OSM building footprints for this block ---
    # IMPORTANT: OSMFetcher bbox order is (left, bottom, right, top)
    # which corresponds to (lon_min, lat_min, lon_max, lat_max)
    fetcher = OSMFetcher(
        bbox=(meta.lon_min, meta.lat_min, meta.lon_max, meta.lat_max),
        tags=OSM_TAGS["buildings"],
    )
    buildings_gdf = fetcher.get_filtered_features(OSM_TAGS["buildings"])
    
    if buildings_gdf.empty:
        print(f"  [buildings] No building footprints found in {block_name}")
    else:
        # --- Rasterize building footprints ---
        rasterizer = BuildingRasterizer(
            gdf=buildings_gdf,
            block_meta=meta,
            resolution_m=RM_RESOLUTION_M,
        )
        building_map = rasterizer.rasterize_with_presence()
    
    # --- Write to HDF5 (only if file exists from Step 3) ---
    h5_path = DATASET_DIR / f"{block_name}.h5"
    
    if h5_path.exists():
        H5Manager.write_dataset(
            h5_path,
            H5Manager.DATASET_BUILDINGS,
            building_map,
            dtype='uint8',
        )
        print(f"  [h5] Written buildings to {h5_path}")
    else:
        print(f"  [h5] Skipped — no H5 file for {block_name} (radio map missing)")

print(f"\n{'='*60}")
print(f"All {len(all_blocks)} blocks processed.")

# Step 5: Transmitter Raster Generation

In this step, we rasterize transmitter locations into a binary presence matrix at the same resolution as the radio map. Each cell is 1 if a transmitter is located within it, 0 otherwise. Transmitters are filtered using the block's **core area** (overlap excluded) to avoid duplicate counting across neighboring blocks, matching the filtering logic in Step 3.

**Key features:**
* Load transmitter locations from `TRANSMITTER_PATH`, the same global file used in Step 3.
* Filter transmitters to only those within the block's **core bounds** (`lat_min_core`, `lat_max_core`, `lon_min_core`, `lon_max_core`), consistent with Step 3's `RadioMapGenerator`.
* Project transmitter geographic coordinates (WGS84) to Lambert-93 planar coordinates using `pyproj.Transformer`.
* Map each transmitter to a single grid cell in a 2D matrix at `RM_RESOLUTION_M` resolution, with the overlap region cropped to match the core area extent.
* Output a **binary presence matrix** (1 = at least one transmitter in cell, 0 = empty cell) with shape matching the radio map and building raster from previous steps.

In [ ]:
from utils.transmitter_mapper import TransmitterMapper

df_tx_all = pd.read_csv(TRANSMITTER_PATH)

for block_info in all_blocks:
    row = block_info["row"]
    col = block_info["col"]
    block_name = block_info["name"]
    
    print(f"\n{'='*60}")
    print(f"Processing {block_name} (row={row}, col={col})")
    print(f"{'='*60}")
    
    # --- Get block metadata ---
    meta = splitter.get_block_latlon_bounds(row, col)
    
    # --- Initialize mapper and filter transmitters ---
    mapper = TransmitterMapper(
        block_meta=meta,
        resolution_m=RM_RESOLUTION_M,
    )
    
    # filter_transmitters uses core bounds (lat_min_core, etc.)
    # to match Step 3 behavior and avoid overlap duplicates
    df_tx_block = mapper.filter_transmitters(df_tx_all)
    
    if len(df_tx_block) == 0:
        print(f"  [transmitters] No transmitters in core area of {block_name}")
    else:
        # --- Rasterize transmitter locations ---
        tx_map = mapper.create_presence_matrix()
    
        # --- Write to HDF5 (only if file exists from Step 3) ---
        h5_path = DATASET_DIR / f"{block_name}.h5"
        
        if h5_path.exists():
            H5Manager.write_dataset(
                h5_path,
                H5Manager.DATASET_TRANSMITTERS,
                tx_map,
                dtype='uint8',
            )
            print(f"  [h5] Written transmitters to {h5_path}")
        else:
            print(f"  [h5] Skipped — no H5 file for {block_name} (radio map missing)")

print(f"\n{'='*60}")
print(f"All {len(all_blocks)} blocks processed.")

# Step 6: Merge Block HDF5 Files

In this step, we merge all individual block HDF5 files (`block_r_c.h5`) from previous steps into a single unified HDF5 file. Each source file is split into smaller blocks of size `MERGED_BLOCK_SIZE_M × MERGED_BLOCK_SIZE_M`, and blocks where the transmitters dataset is entirely zero are discarded. The valid samples from all source files are stacked into a single output file with shape `(N, MERGED_BLOCK_SIZE_M, MERGED_BLOCK_SIZE_M)`.

**Key features:**
* Scan `DATASET_DIR` for all files matching the pattern `block_\d+_\d+\.h5`.
* Skip files with `has_simulation=False` (incomplete data) and warn.
* Validate that `resolution_m` and `overlap_m` are consistent across all source files; raise an error if they differ.
* Verify that each source file's `block_size_m` is **divisible** by `MERGED_BLOCK_SIZE_M` (old ≥ new, old % new == 0); raise an error otherwise.
* Split the three datasets (`buildings`, `transmitters`, `radiomap`) into non-overlapping windows of `MERGED_BLOCK_SIZE_M × MERGED_BLOCK_SIZE_M`.
* Discard any window where the **entire** transmitters block is zero (no transmitters present).
* Stack all valid windows along a new sample dimension `N`, preserving dtype (`uint8` for buildings/transmitters, `float32` for radiomap).
* Write the merged output to `MERGED_DIR / MERGED_FILENAME` with compression enabled.
* If no valid samples remain after filtering, the output file is **not created**.

**Output file structure:**

```
file: block.h5
├── Attributes
│   ├── block_name: "block"          (str)
│   ├── block_size_m: 256            (int)
│   ├── resolution_m: 1.0            (float)   ← inherited from source files
│   ├── overlap_m: 150               (int)     ← inherited (for traceability)
│   ├── has_simulation: True         (bool)
│   └── dataset_size: N              (uint16)  ← number of valid samples
│
└── Datasets ([N, block_size_m, block_size_m])
    ├── /buildings     --> dtype: uint8   | buildings map
    ├── /transmitters  --> dtype: uint8   | transmitters map
    └── /radiomap      --> dtype: float32 | radio map
```

**Parameters:**

| Variable | Description |
|----------|-------------|
| `MERGED_BLOCK_SIZE_M` | Target block size for each sample (default: 256) |
| `MERGED_DIR` | Output directory for the merged file |
| `MERGED_FILENAME` | Name of the merged output file (default: `block.h5`) |
| `DATASET_DIR` | Directory containing source `block_r_c.h5` files |
```

In [ ]:
# Step 6: Merge Block HDF5 Files
MERGED_BLOCK_SIZE_M = 256
MERGED_FILENAME = Path("data/merged_dataset/block.h5")

In [ ]:
# Step 6: Merge Block HDF5 Files
import h5py
from utils.h5_manager import H5Manager

# Ensure output directory exists
MERGED_FILENAME.parent.mkdir(parents=True, exist_ok=True)

print(f"Merging blocks from: {DATASET_DIR}")
print(f"Target block size:   {MERGED_BLOCK_SIZE_M} m")
print(f"Output:              {MERGED_FILENAME}")

H5Manager.merge_blocks(
    input_dir=DATASET_DIR,
    new_block_size_m=MERGED_BLOCK_SIZE_M,
    output_path=MERGED_FILENAME,
)

if MERGED_FILENAME.exists():
    with h5py.File(MERGED_FILENAME, "r") as f:
        N = f.attrs["dataset_size"]
        print(f"\nMerged {N} samples into {MERGED_FILENAME}")
        print(f"  buildings:    {f['buildings'].shape}  | dtype={f['buildings'].dtype}")
        print(f"  transmitters: {f['transmitters'].shape}  | dtype={f['transmitters'].dtype}")
        print(f"  radiomap:     {f['radiomap'].shape}  | dtype={f['radiomap'].dtype}")
else:
    print("\nNo output file created (no valid samples).")